# Inspect saved pipeline results
Choose **Run → Run All Cells**. This notebook reads saved files on CPU; it does not run LingBot, SAM, Qwen, or evaluation. The kernel needs NumPy, Pillow, and Matplotlib.

On KTH it searches `/home/jovyan/EmbodiedAI-pipelines/outputs`. Locally it searches `outputs/pipeline_fixture_v1`. It selects the newest real `geometry_only` run, or a fixture when only fixtures exist. The selection and all available runs appear below.

To inspect another result, set `RUN_DIR` in the next cell. Relative paths resolve from the project root shown there. For a relocated geometry cache, set `CACHE_DIR` to its directory; the saved archive hash must match. To show the provenance of an existing evaluation, set `REPORT_PATH` to its `report.json`.

**Fixture figures are synthetic software checks, not real accuracy or performance results. Without an independent reference evaluation, these plots support inspection only.**

In [ ]:
from pathlib import Path
import sys

candidates = [Path.cwd(), *Path.cwd().parents, Path('/home/jovyan/EmbodiedAI-pipelines')]
REPO = next((path for path in candidates if (path / 'src/inspect_pipeline_results.py').is_file()), None)
if REPO is None:
    raise FileNotFoundError('Open this notebook from the project checkout containing src/inspect_pipeline_results.py')
sys.path.insert(0, str(REPO / 'src'))
from inspect_pipeline_results import default_outputs_root, discover_runs, choose_run, load_run, summary_text, create_figures, export_html

ROOT = default_outputs_root(REPO)
RUN_DIR = None       # Example: 'outputs/pipeline_fixture_v1/runs/fixed_hazards'
CACHE_DIR = None     # Optional relocated directory containing geometry.npz
REPORT_PATH = None   # Optional saved independent-reference report.json
MAX_POINTS = 20000   # Upper bound for each plotted 3D point sample
MAX_CONCEPTS = 16    # Raise to inspect additional concepts

def resolve_input(value):
    if value is None:
        return None
    path = Path(value).expanduser()
    return path.resolve() if path.is_absolute() else (REPO / path).resolve()

ROOT = resolve_input(ROOT)
print(f'Project: {REPO}\nSearch root: {ROOT}\n')
for row in discover_runs(ROOT):
    print(f"{row['pipeline']:16} status={row['status']:12} fixture={row['fixture']}  {row['path']}")
selected = resolve_input(RUN_DIR) if RUN_DIR is not None else choose_run(ROOT)
print(f'\nSelected: {selected}')


In [ ]:
result = load_run(selected, cache_dir=resolve_input(CACHE_DIR), report_path=resolve_input(REPORT_PATH))
print(summary_text(result))


The processed RGB thumbnails show the camera input grid used by geometry. In the 3D plot, source RGB colors come from matching saved world points; gray points are a voxel-center fallback. Red markers are saved camera centers. The axes retain the recorded map frame and units: the viewer does not infer gravity, assume an XY ground plane, or apply scale again.

Semantic plots show positive stored evidence per concept. Their color is a weighted evidence statistic, not a calibrated probability or safety certificate. Missing evidence does not mean safe ground. `geometry_only` has no semantic model, so it normally has no semantic plots.

The robot-policy grid is an internal diagnostic: gray means unknown, red blocked, and green traversable under the recorded policy. Read the planning status and blocked reason above; the grid does not establish successful navigation or collision-free robot operation. A partial run can have incomplete geometry and evidence.

In [ ]:
from IPython.display import display
import matplotlib.pyplot as plt

warnings_before = len(result['warnings'])
for title, figure in create_figures(result, max_points=MAX_POINTS, max_concepts=MAX_CONCEPTS):
    print(title)
    display(figure)
    plt.close(figure)
for warning in result['warnings'][warnings_before:]:
    print('Note:', warning)


For a standalone saved view, uncomment the line below and run that cell. It creates `report.html` with embedded static figures in a separate folder; it leaves the producer run untouched. This is an inspection report, not a new evaluation.

In [ ]:
# print(export_html(result, REPO / 'outputs/pipeline_results_preview', max_points=MAX_POINTS))
